# Random Forest Project
For this project we will be exploring publicly available data from [LendingClub.com](https://www.lendingclub.com/). Lending Club connects people who need money (borrowers) with people who have money (investors). Hopefully, as an investor you would want to invest in people who showed a profile of having a high probability of paying you back. We will try to create a model that will help predict this.

Lending club had a [very interesting year in 2016](https://en.wikipedia.org/wiki/Lending_Club#2016), so let's check out some of their data and keep the context in mind. This data is from before they even went public.

We will use lending data from 2007-2010 and be trying to classify and predict whether or not the borrower paid back their loan in full. You can use the csv already provided on Blackboard. The csv provided has been cleaned of NA values.

Here are what the columns represent:
* credit.policy: 1 if the customer meets the credit underwriting criteria of LendingClub.com, and 0 otherwise.
* purpose: The purpose of the loan (takes values "credit_card", "debt_consolidation", "educational", "major_purchase", "small_business", and "all_other").
* int.rate: The interest rate of the loan, as a proportion (a rate of 11% would be stored as 0.11). Borrowers judged by LendingClub.com to be more risky are assigned higher interest rates.
* installment: The monthly installments owed by the borrower if the loan is funded.
* log.annual.inc: The natural log of the self-reported annual income of the borrower.
* dti: The debt-to-income ratio of the borrower (amount of debt divided by annual income).
* fico: The FICO credit score of the borrower.
* days.with.cr.line: The number of days the borrower has had a credit line.
* revol.bal: The borrower's revolving balance (amount unpaid at the end of the credit card billing cycle).
* revol.util: The borrower's revolving line utilization rate (the amount of the credit line used relative to total credit available).
* inq.last.6mths: The borrower's number of inquiries by creditors in the last 6 months.
* delinq.2yrs: The number of times the borrower had been 30+ days past due on a payment in the past 2 years.
* pub.rec: The borrower's number of derogatory public records (bankruptcy filings, tax liens, or judgments).


## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Get the Data

In [ ]:
loans = pd.read_csv('loan_data.csv')
loans.head()

In [ ]:
loans.info()

In [ ]:
loans.describe()

# Exploratory Data Analysis

### FICO Distribution by `credit.policy`

In [ ]:
plt.figure(figsize=(10,5))
loans[loans['credit.policy']==1]['fico'].hist(bins=35, color='blue', alpha=0.6, label='Credit Policy = 1')
loans[loans['credit.policy']==0]['fico'].hist(bins=35, color='red',  alpha=0.6, label='Credit Policy = 0')
plt.legend()
plt.xlabel('FICO Score')
plt.ylabel('Count')
plt.title('FICO Distribution by Credit Policy')
plt.show()

### FICO Distribution by `not.fully.paid`

In [ ]:
plt.figure(figsize=(10,5))
loans[loans['not.fully.paid']==1]['fico'].hist(bins=35, color='red',  alpha=0.6, label='not.fully.paid = 1')
loans[loans['not.fully.paid']==0]['fico'].hist(bins=35, color='blue', alpha=0.6, label='not.fully.paid = 0')
plt.legend()
plt.xlabel('FICO Score')
plt.ylabel('Count')
plt.title('FICO Distribution by Loan Repayment Status')
plt.show()

### Countplot: Loan Purpose by `not.fully.paid`

In [ ]:
plt.figure(figsize=(12,5))
sns.countplot(x='purpose', hue='not.fully.paid', data=loans, palette='Set1')
plt.title('Loan Count by Purpose and Repayment Status')
plt.xticks(rotation=45)
plt.show()

### Jointplot: FICO Score vs Interest Rate

In [ ]:
sns.jointplot(x='fico', y='int.rate', data=loans, color='purple', alpha=0.3)
plt.suptitle('FICO Score vs Interest Rate', y=1.02)
plt.show()

### lmplot: FICO vs Interest Rate split by `credit.policy` and `not.fully.paid`

In [ ]:
sns.lmplot(x='fico', y='int.rate', data=loans,
           hue='not.fully.paid', col='credit.policy',
           palette='Set1', scatter_kws={'alpha': 0.3})
plt.suptitle('FICO vs Interest Rate by Credit Policy', y=1.02)
plt.show()

# Setting Up the Data

### Handle Categorical Feature: `purpose`

In [ ]:
cat_feats = ['purpose']
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)
final_data.head()

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

## Training a Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

## Predictions and Evaluation – Decision Tree

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

dt_pred = dtree.predict(X_test)

print("=== Classification Report – Decision Tree ===")
print(classification_report(y_test, dt_pred))

print("=== Confusion Matrix – Decision Tree ===")
print(confusion_matrix(y_test, dt_pred))

## Training the Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=600, random_state=42)
rfc.fit(X_train, y_train)

## Predictions and Evaluation – Random Forest

In [ ]:
rfc_pred = rfc.predict(X_test)

print("=== Classification Report – Random Forest ===")
print(classification_report(y_test, rfc_pred))

print("=== Confusion Matrix – Random Forest ===")
print(confusion_matrix(y_test, rfc_pred))

## What Performed Better?

The **Random Forest** outperforms the single Decision Tree:

- **Decision Tree** tends to overfit — it memorizes training data but generalises poorly.
- **Random Forest** (600 trees) reduces variance by averaging many decorrelated trees, giving higher accuracy overall.
